# 🎤 NB-09: Whisper Fine-tuning – TTS→ASR on Claude Responses
**Goal:** Synthesize audio from Claude responses using pyttsx3, then fine-tune Whisper-tiny for domain-specific ASR.

**Modality:** Speech/Audio | **Model:** openai/whisper-tiny

In [ ]:
!pip install transformers datasets librosa soundfile pyttsx3 evaluate jiwer -q

In [ ]:
import json, re, random
random.seed(42)

def load_dataset(path="claude-opus-4_5-250x_jsonl.txt"):
    """Load Claude thinking dataset from JSONL."""
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

def extract_fields(records):
    data = []
    for r in records:
        msgs = r["messages"]
        user = next((m["content"] for m in msgs if m["role"]=="user"), "")
        asst = next((m["content"] for m in msgs if m["role"]=="assistant"), "")
        think = re.findall(r"<think>(.*?)</think>", asst, re.DOTALL)
        response = re.sub(r"<think>.*?</think>", "", asst, flags=re.DOTALL).strip()
        data.append({
            "user": user,
            "assistant_full": asst,
            "thinking": " ".join(think).strip(),
            "response": response
        })
    return data

records = load_dataset()
data = extract_fields(records)
print(f"Loaded {len(data)} examples")
print(f"Sample user: {data[0]['user'][:80]}")
print(f"Sample think: {data[0]['thinking'][:120]}...")
print(f"Sample resp: {data[0]['response'][:120]}...")


In [ ]:
import pyttsx3, soundfile as sf, numpy as np, os, tempfile

engine = pyttsx3.init()
engine.setProperty("rate", 150)

audio_files, transcripts = [], []
os.makedirs("./audio_data", exist_ok=True)

print("Synthesizing audio from Claude responses...")
for i, d in enumerate(data[:50]):  # 50 samples for demo
    txt = d["response"][:200].replace("\n", " ").strip()
    if not txt: continue
    path = f"./audio_data/{i:03d}.wav"
    engine.save_to_file(txt, path)
    engine.runAndWait()
    if os.path.exists(path):
        audio_files.append(path)
        transcripts.append(txt)

print(f"Synthesized {len(audio_files)} audio files")


In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer
from datasets import Dataset, Audio
import torch

processor = WhisperProcessor.from_pretrained("openai/whisper-tiny")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny")
model.config.forced_decoder_ids = None

ds = Dataset.from_dict({"audio": audio_files, "text": transcripts})
ds = ds.cast_column("audio", Audio(sampling_rate=16000)).train_test_split(0.1)

def prepare(batch):
    audio = batch["audio"]["array"]
    batch["input_features"] = processor.feature_extractor(audio, sampling_rate=16000).input_features[0]
    batch["labels"] = processor.tokenizer(batch["text"]).input_ids
    return batch

ds = ds.map(prepare, remove_columns=ds["train"].column_names)

args = Seq2SeqTrainingArguments("./whisper-claude", num_train_epochs=5,
    per_device_train_batch_size=4, predict_with_generate=True,
    fp16=torch.cuda.is_available(), evaluation_strategy="epoch", report_to="none")
Seq2SeqTrainer(model=model, args=args, train_dataset=ds["train"],
    eval_dataset=ds["test"], tokenizer=processor.feature_extractor).train()
print("✅ Whisper ASR fine-tuned!")
